# Regularización

Compara una red pequeña y redes grandes con y sin L2, Dropout y parada temprana. La comparación usa el conjunto de desarrollo.


# Preparación de datos MNIST

Este bloque se incluye en cada notebook para que pueda ejecutarse de forma independiente.

In [ ]:
# Librerías, semilla y carga de MNIST
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, "Dev:", x_dev.shape, "Test:", x_test.shape)


In [ ]:
# REDUCCIÓN DEL CONJUNTO DE ENTRENAMIENTO PARA EL EXPERIMENTO DE REGULARIZACIÓN
# Se toma una muestra estratificada de 12 000 imágenes del conjunto de entrenamiento.
# Esto hace más rápido el experimento y conserva aproximadamente la distribución de las clases.

x_train_reg, _, y_train_reg, _ = train_test_split(

    x_train, y_train,

    train_size=12000,

    random_state=SEED,

    stratify=y_train

)



print("Muestra:", x_train_reg.shape, y_train_reg.shape)

In [ ]:
# MODELO GRANDE SIN REGULARIZACIÓN
# Se construye una red con dos capas ocultas de 512 y 256 neuronas.
# No se aplican técnicas explícitas de regularización.
# El objetivo es observar si un modelo con mayor capacidad presenta sobreajuste.
# Se entrena durante 15 épocas y se guarda la historia para compararla posteriormente.

def crear_modelo_sin_regularizacion():

    modelo = keras.Sequential([

        layers.Input(shape=(28, 28)),

        layers.Flatten(),

        layers.Dense(512, activation="relu"),

        layers.Dense(256, activation="relu"),

        layers.Dense(10, activation="softmax")

    ])

    modelo.compile(

        optimizer=keras.optimizers.Adam(0.001),

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return modelo



modelo_sin_reg = crear_modelo_sin_regularizacion()

hist_sin_reg = modelo_sin_reg.fit(

    x_train_reg, y_train_reg,

    validation_data=(x_dev, y_dev),

    epochs=15,

    batch_size=64,

    verbose=0

)


In [ ]:
# MODELO REGULARIZADO
# Se mantiene una arquitectura similar, pero se incorpora regularización L2 y Dropout.
# L2 penaliza pesos demasiado grandes durante el entrenamiento.
# Dropout desactiva aleatoriamente el 30 % de activaciones durante el entrenamiento.
# EarlyStopping detiene el entrenamiento cuando la validation loss deja de mejorar.
# restore_best_weights recupera los pesos correspondientes al mejor punto de validación.

def crear_modelo_regularizado():

    modelo = keras.Sequential([

        layers.Input(shape=(28, 28)),

        layers.Flatten(),

        layers.Dense(

            512, activation="relu",

            kernel_regularizer=regularizers.l2(1e-4)

        ),

        layers.Dropout(0.30),

        layers.Dense(

            256, activation="relu",

            kernel_regularizer=regularizers.l2(1e-4)

        ),

        layers.Dropout(0.30),

        layers.Dense(10, activation="softmax")

    ])

    modelo.compile(

        optimizer=keras.optimizers.Adam(0.001),

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return modelo



parada_temprana = callbacks.EarlyStopping(

    monitor="val_loss",

    patience=2,

    restore_best_weights=True

)



modelo_reg = crear_modelo_regularizado()

hist_reg = modelo_reg.fit(

    x_train_reg, y_train_reg,

    validation_data=(x_dev, y_dev),

    epochs=20,

    batch_size=64,

    callbacks=[parada_temprana],

    verbose=0

)



print("Épocas ejecutadas:", len(hist_reg.history["loss"]))

In [ ]:
# COMPARACIÓN DE LAS CURVAS DE REGULARIZACIÓN
# Se muestran las pérdidas de entrenamiento y desarrollo de ambos modelos.
# Una brecha creciente entre train loss y dev loss puede indicar sobreajuste.
# La comparación permite observar si L2, Dropout y EarlyStopping ayudan a generalizar mejor.

plt.figure(figsize=(8, 5))

plt.plot(hist_sin_reg.history["loss"], label="Train sin regularización")

plt.plot(hist_sin_reg.history["val_loss"], label="Dev sin regularización")

plt.plot(hist_reg.history["loss"], label="Train regularizado")

plt.plot(hist_reg.history["val_loss"], label="Dev regularizado")

plt.xlabel("Época")

plt.ylabel("Loss")

plt.title("Regularización: comparación de pérdidas")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# EVALUACIÓN EN EL CONJUNTO DE DESARROLLO
# Se calculan loss y accuracy del modelo sin regularización y del regularizado.
# Los resultados permiten comparar el desempeño de ambos enfoques fuera de los datos usados para ajustar los pesos.

dev_sin_reg = modelo_sin_reg.evaluate(x_dev, y_dev, verbose=0)

dev_reg = modelo_reg.evaluate(x_dev, y_dev, verbose=0)



print("Sin regularización - dev loss/accuracy:", dev_sin_reg)

print("Regularizado      - dev loss/accuracy:", dev_reg)



In [ ]:
# MODELO PEQUEÑO
# Se crea una red mucho más pequeña con solamente 16 neuronas en la capa oculta.
# El experimento sirve para evaluar el efecto de reducir la capacidad del modelo.
# Se entrena durante solo 2 épocas para obtener una referencia rápida de desempeño.

modelo_pequeno = keras.Sequential([

    layers.Input(shape=(28, 28)),

    layers.Flatten(),

    layers.Dense(16, activation="relu"),

    layers.Dense(10, activation="softmax")

])



modelo_pequeno.compile(

    optimizer=keras.optimizers.Adam(0.001),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)



hist_pequeno = modelo_pequeno.fit(

    x_train, y_train,

    validation_data=(x_dev, y_dev),

    epochs=2,

    batch_size=64,

    verbose=0

)

In [ ]:
# FUNCIÓN PARA DIAGNOSTICAR LA BRECHA ENTRE TRAIN Y DEV
# La función obtiene el accuracy final de entrenamiento y validación.
# La brecha train-dev mide cuánto mejor se comporta el modelo sobre los datos de entrenamiento.
# Una brecha grande puede ser una señal de sobreajuste.

def resumen_historia(nombre, historia):

    train_acc = historia.history["accuracy"][-1]

    dev_acc = historia.history["val_accuracy"][-1]

    return {

        "modelo": nombre,

        "train_accuracy": train_acc,

        "dev_accuracy": dev_acc,

        "brecha_train_dev": train_acc - dev_acc

    }



tabla_diagnostico = pd.DataFrame([

    resumen_historia("Pequeño", hist_pequeno),

    resumen_historia("Sin regularización", hist_sin_reg),

    resumen_historia("Regularizado", hist_reg)

])



display(tabla_diagnostico)